# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - INFORMACIÓN GOBERNANZA

## **M0. Configuración General**

In [1]:
import pandas as pd
import requests
from io import BytesIO

# URLs de tus archivos en GitHub
archivos = {
    "servicios": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/SERVICIOS_PUBLICOS.xlsx",
    "economia": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/ECONOMIA.xlsx",
    "descripcion": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/DESCRIPCION_GENERAL.xlsx"
}

municipios_framework = [
    "CHIA",
    "SOPO",
    "TOCANCIPA",
    "VILLAPINZON"
]

def validar_estructura(url, nombre):
    try:
        # Descarga el archivo
        response = requests.get(url)
        if response.status_code == 200:
            file_data = BytesIO(response.content)

            # Leemos las primeras filas sin encabezado para identificar la estructura
            df_raw = pd.read_excel(file_data, header=None)

            # Buscamos la fila donde está 'MUNICIPIO' (basado en tu imagen)
            header_idx = -1
            for i in range(15): # Escaneamos las primeras 15 filas
                if 'MUNICIPIO' in df_raw.iloc[i].values:
                    header_idx = i
                    break

            if header_idx != -1:
                print(f"\n✅ Archivo '{nombre}' validado correctamente.")
                print(f"   -> Encabezados detectados en la fila: {header_idx}")

                # Cargamos el archivo usando la fila correcta como encabezado
                df_final = pd.read_excel(BytesIO(response.content), header=header_idx)

                # Resumen de validación
                print(f"   -> Columnas encontradas: {list(df_final.columns)}")
                print(f"   -> Total de registros: {len(df_final)}")
                return df_final
            else:
                print(f"\n❌ Error en '{nombre}': No se encontró la columna 'MUNICIPIO'.")
                return None
        else:
            print(f"\n❌ Error al conectar con GitHub ({nombre}). Código: {response.status_code}")
            return None
    except Exception as e:
        print(f"\n❌ Error técnico procesando '{nombre}': {e}")
        return None

## **M1. Definición de la Fuente Oficial**

In [2]:
# Ejecución de la revisión
dfs = {}
for nombre, url in archivos.items():
    dfs[nombre] = validar_estructura(url, nombre)


✅ Archivo 'servicios' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 7426

✅ Archivo 'economia' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 577

✅ Archivo 'descripcion' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 3269


## **M2. Extracción y Procesamiento de Datos**

In [3]:
indicadores_servicios = [
    "COBERTURA DE ACUEDUCTO URBANO",
    "COBERTURA DE ACUEDUCTO RURAL",
    "COBERTURA DE ALCANTARILLADO URBANO",
    "COBERTURA DE ALCANTARILLADO RURAL",
    "ACCESO A AGUA POTABLE ADECUADO",
    "PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS",
    "CALIDAD DEL AGUA",
    "CONTINUIDAD DE ACUEDUCTO URBANO",
    "AGUAS RESIDUALES TRATADAS"
]

indicadores_economia = [
    "PRODUCTO INTERNO BRUTO DEPARTAMENTAL"
]

indicadores_descripcion = [
    "POBLACION TOTAL",
    "DENSIDAD POBLACIONAL",
    "PORCENTAJE POBLACION URBANA",
    "PORCENTAJE POBLACION RURAL",
    "INDICE DE DESEMPENO INSTITUCIONAL"
]

# Filtrar los dataframes de acuerdo a los indicadores seleccionados
servicios = dfs['servicios'][
    dfs['servicios']["INDICADOR"].isin(indicadores_servicios)
].copy()

economia = dfs['economia'][
    dfs['economia']["INDICADOR"].isin(indicadores_economia)
].copy()

descripcion = dfs['descripcion'][
    dfs['descripcion']["INDICADOR"].isin(indicadores_descripcion)
].copy()

print("Servicios (filtrado):", servicios.shape)
print("Economía (filtrado):", economia.shape)
print("Descripción (filtrado):", descripcion.shape)

Servicios (filtrado): (3480, 12)
Economía (filtrado): (569, 12)
Descripción (filtrado): (2228, 12)


In [4]:
# Concatenar todos los dataframes filtrados
gobernanza = pd.concat(
    [
        servicios,
        economia,
        descripcion
    ],
    ignore_index=True
)

# Filtrar por los municipios de interés
gobernanza_filtrada = gobernanza[
    gobernanza["MUNICIPIO"].isin(municipios_framework)
].copy()

print("Dimensiones del DataFrame consolidado y filtrado:", gobernanza_filtrada.shape)

Dimensiones del DataFrame consolidado y filtrado: (216, 12)


In [5]:
# Transformar a formato ancho (pivot_table)
gobernanza_matriz = gobernanza_filtrada.pivot_table(
    index=["COD_MUN", "MUNICIPIO", "YEAR"],
    columns="INDICADOR",
    values="DATO_NUMERICO",
    aggfunc="first"
).reset_index()

# Reordenar columnas a la secuencia deseada
gobernanza_matriz = gobernanza_matriz[[
    "COD_MUN",
    "MUNICIPIO",
    "YEAR",
    "POBLACION TOTAL",
    "DENSIDAD POBLACIONAL",
    "PORCENTAJE POBLACION URBANA",
    "PORCENTAJE POBLACION RURAL",
    "PRODUCTO INTERNO BRUTO DEPARTAMENTAL",
    "INDICE DE DESEMPENO INSTITUCIONAL",
    "ACCESO A AGUA POTABLE ADECUADO",
    "PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS",
    "COBERTURA DE ACUEDUCTO URBANO",
    "COBERTURA DE ACUEDUCTO RURAL",
    "COBERTURA DE ALCANTARILLADO URBANO",
    "COBERTURA DE ALCANTARILLADO RURAL",
    "CONTINUIDAD DE ACUEDUCTO URBANO",
    "CALIDAD DEL AGUA",
    "AGUAS RESIDUALES TRATADAS"
]]

# Ordenar por municipio y año
gobernanza_matriz = gobernanza_matriz.sort_values(
    ["MUNICIPIO", "YEAR"]
).reset_index(drop=True)

# Renombrar columna 'YEAR' a 'Anio'
gobernanza_matriz.rename(
    columns={
        "YEAR": "Anio"
    },
    inplace=True
)

# Insertar columna 'Nodo' al inicio con el nombre del municipio
gobernanza_matriz.insert(
    0,
    "Nodo",
    gobernanza_matriz["MUNICIPIO"]
)

print("Dimensiones de la matriz de gobernanza final:", gobernanza_matriz.shape)
display(gobernanza_matriz.head())

Dimensiones de la matriz de gobernanza final: (20, 19)


INDICADOR,Nodo,COD_MUN,MUNICIPIO,Anio,POBLACION TOTAL,DENSIDAD POBLACIONAL,PORCENTAJE POBLACION URBANA,PORCENTAJE POBLACION RURAL,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,INDICE DE DESEMPENO INSTITUCIONAL,ACCESO A AGUA POTABLE ADECUADO,PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS,COBERTURA DE ACUEDUCTO URBANO,COBERTURA DE ACUEDUCTO RURAL,COBERTURA DE ALCANTARILLADO URBANO,COBERTURA DE ALCANTARILLADO RURAL,CONTINUIDAD DE ACUEDUCTO URBANO,CALIDAD DEL AGUA,AGUAS RESIDUALES TRATADAS
0,CHIA,25175,CHIA,2020,NaN,NaN,83.11,16.89,3503.635640,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CHIA,25175,CHIA,2021,148415.0,1952.83,83.72,16.28,4194.505649,NaN,100.0,99.00,100.0,100.0,84.66,61.07,NaN,NaN,NaN
2,CHIA,25175,CHIA,2022,153130.0,2014.87,84.17,15.83,160.198863,NaN,100.0,99.00,1.0,100.0,98.09,59.83,NaN,NaN,NaN
3,CHIA,25175,CHIA,2023,158258.0,2082.34,84.58,15.42,5711.022704,NaN,100.0,98.85,1.0,100.0,99.00,1.00,24.0,25.00,69.0
4,CHIA,25175,CHIA,2024,163306.0,2148.76,84.97,15.03,6182.859140,69.6,100.0,98.85,1.0,100.0,99.00,1.00,24.0,36.32,69.0


## **M3. Reporte de Auditoría**

In [6]:
audit_dfs = {}

for nombre_df_original, df_original in dfs.items():
    df_temp = df_original.copy()
    df_temp["MUNICIPIO"] = df_temp["MUNICIPIO"].str.upper()
    filtro_municipios = df_temp[df_temp["MUNICIPIO"].isin(municipios_framework)]

    # Auditoría 1: Resumen de indicadores y años
    resumen_indicadores = (
        filtro_municipios.groupby("INDICADOR")
        .agg(
            Año_Inicial=("YEAR", "min"),
            Año_Final=("YEAR", "max"),
            Registros=("YEAR", "count"),
            Municipios=("MUNICIPIO", "nunique")
        )
        .sort_values("Registros", ascending=False)
    )
    audit_dfs[f'{nombre_df_original}_ResumenIndicadores'] = resumen_indicadores

    # Auditoría 2: Información por municipio
    info_municipio = filtro_municipios.groupby(["INDICADOR", "MUNICIPIO"]).size().unstack(fill_value=0)
    audit_dfs[f'{nombre_df_original}_InfoMunicipio'] = info_municipio

    # Auditoría 3: Valores faltantes (pivot_table de conteo)
    valores_faltantes = pd.pivot_table(
        filtro_municipios,
        index="INDICADOR",
        columns="MUNICIPIO",
        values="DATO_NUMERICO",
        aggfunc="count"
    ).fillna(0)
    audit_dfs[f'{nombre_df_original}_ValoresFaltantes'] = valores_faltantes

# Crear un resumen de la matriz de gobernanza final
final_matrix_summary = gobernanza_matriz.describe().transpose()
audit_dfs['MatrizGobernanza_Summary'] = final_matrix_summary

print("Reportes de auditoría generados y listos para exportar.")

Reportes de auditoría generados y listos para exportar.


## **M4. Exportación de Resultados**

In [7]:
# Exportar la matriz de gobernanza final
gobernanza_matriz.to_excel(
    "01_Capa_Gobernanza_V1.xlsx",
    index=False
)
print("Archivo '01_Capa_Gobernanza_V1.xlsx' generado correctamente.")

# Exportar los reportes de auditoría a un solo archivo Excel con múltiples hojas
with pd.ExcelWriter('Auditoria_Capa_Gobernanza.xlsx') as writer:
    for sheet_name, df_audit in audit_dfs.items():
        df_audit.to_excel(writer, sheet_name=sheet_name)
print("Archivo 'Auditoria_Capa_Gobernanza.xlsx' generado correctamente.")

Archivo '01_Capa_Gobernanza_V1.xlsx' generado correctamente.
Archivo 'Auditoria_Capa_Gobernanza.xlsx' generado correctamente.


## **M5. Resumen Final**

A diferencia de la capa Climática (mensual) o la Hidráulica (mensual), esta capa de Gobernanza es anual. Esto no es un problema; simplemente significa que en la integración del Framework cada valor anual se asociará al periodo correspondiente mediante el campo Anio. Más adelante, cuando construyamos el dataset maestro para los modelos LSTM, definiremos la estrategia para armonizar las distintas resoluciones temporales (mensual vs. anual) sin perder consistencia.

In [8]:
import pandas as pd
import requests
from io import BytesIO

# URLs de tus archivos en GitHub
archivos = {
    "servicios": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/SERVICIOS_PUBLICOS.xlsx",
    "economia": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/ECONOMIA.xlsx",
    "descripcion": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/DESCRIPCION_GENERAL.xlsx"
}

def validar_estructura(url, nombre):
    try:
        # Descarga el archivo
        response = requests.get(url)
        if response.status_code == 200:
            file_data = BytesIO(response.content)

            # Leemos las primeras filas sin encabezado para identificar la estructura
            df_raw = pd.read_excel(file_data, header=None)

            # Buscamos la fila donde está 'MUNICIPIO' (basado en tu imagen)
            header_idx = -1
            for i in range(15): # Escaneamos las primeras 15 filas
                if 'MUNICIPIO' in df_raw.iloc[i].values:
                    header_idx = i
                    break

            if header_idx != -1:
                print(f"\n✅ Archivo '{nombre}' validado correctamente.")
                print(f"   -> Encabezados detectados en la fila: {header_idx}")

                # Cargamos el archivo usando la fila correcta como encabezado
                df_final = pd.read_excel(BytesIO(response.content), header=header_idx)

                # Resumen de validación
                print(f"   -> Columnas encontradas: {list(df_final.columns)}")
                print(f"   -> Total de registros: {len(df_final)}")
                return df_final
            else:
                print(f"\n❌ Error en '{nombre}': No se encontró la columna 'MUNICIPIO'.")
                return None
        else:
            print(f"\n❌ Error al conectar con GitHub ({nombre}). Código: {response.status_code}")
            return None
    except Exception as e:
        print(f"\n❌ Error técnico procesando '{nombre}': {e}")
        return None

# Ejecución de la revisión
dfs = {}
for nombre, url in archivos.items():
    dfs[nombre] = validar_estructura(url, nombre)


✅ Archivo 'servicios' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 7426

✅ Archivo 'economia' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 577

✅ Archivo 'descripcion' validado correctamente.
   -> Encabezados detectados en la fila: 11
   -> Columnas encontradas: ['COD_DEP', 'DEPARTAMENTO', 'COD_MUN', 'MUNICIPIO', 'DIMENSION', 'SUBCATEGORIA', 'INDICADOR', 'DATO_NUMERICO', 'CUALITATIVO', 'YEAR', 'FUENTE', 'UNIDAD_DE_MEDIDA']
   -> Total de registros: 3269


In [9]:
import pandas as pd

# Lista de tus archivos locales
archivos = {
    "SERVICIOS": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/SERVICIOS_PUBLICOS.xlsx",
    "ECONOMIA": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/ECONOMIA.xlsx",
    "DESCRIPCION": "https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/DESCRIPCION_GENERAL.xlsx"
}

for nombre, ruta in archivos.items():
    # Leemos usando la fila 11, que ya confirmamos es la correcta
    df = pd.read_excel(ruta, header=11)

    print(f"\n--- Auditoría de Indicadores en {nombre} ---")
    indicadores = df['INDICADOR'].unique()
    print(f"Total de indicadores únicos: {len(indicadores)}")
    print("Primeros 15 indicadores detectados:")
    for ind in indicadores[:15]:
        print(f"- {ind}")


--- Auditoría de Indicadores en SERVICIOS ---
Total de indicadores únicos: 18
Primeros 15 indicadores detectados:
- PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS
- PORCENTAJE DE AGUAS RESIDUALES INDUSTRIALES TRATADAS
- ACCESO A AGUA POTABLE ADECUADO
- CALIDAD DEL AGUA
- COBERTURA DE ALCANTARILLADO RURAL
- ALCANTARILLADO EN AREA CENTROS POBLADOS
- ALCANTARILLADO EN AREA MUNICIPAL O DISTRITAL
- ALCANTARILLADO EN AREA RURAL (CENTROS POBLADOS Y FINCAS Y VIVIENDAS DISPERSA)
- ALCANTARILLADO EN AREA URBANA
- AGUAS RESIDUALES TRATADAS
- CONTINUIDAD DE ACUEDUCTO URBANO
- COBERTURA DE ACUEDUCTO URBANO
- COBERTURA DE ACUEDUCTO RURAL
- ACUEDUCTO EN AREA CENTROS POBLADOS
- ACUEDUCTO EN AREA MUNICIPAL O DISTRITAL

--- Auditoría de Indicadores en ECONOMIA ---
Total de indicadores únicos: 3
Primeros 15 indicadores detectados:
- PRODUCTO INTERNO BRUTO DEPARTAMENTAL
- EXPORTACIONES DE BIENES NO MINERO ENERGETICOS
- INDICE DE DESEMPENO INSTITUCIONAL

--- Auditoría de Indicado

**Evaluación: Selección de Indicadores para el Framework**

Los tres archivos analizados tienen exactamente la misma estructura:

* **Servicios Públicos:** 7.426 registros
* **Economía:** 1.857 registros
* **Descripción General:** 3.269 registros

Todos contienen los campos: `MUNICIPIO`, `YEAR`, `INDICADOR`, `DATO_NUMERICO` y `CUALITATIVO`. Esto significa que **no son tres datasets diferentes**, sino tres dimensiones de un mismo sistema de indicadores territoriales.

---

**¿Cuál sirve realmente para el Framework?**

Recordemos el objetivo principal: **No estamos construyendo un modelo económico ni demográfico.** Estamos construyendo un **Framework de Gestión Hídrica basado en Ciencia de Datos**.

Por lo tanto, la pregunta fundamental es: *¿Qué variables representan la capacidad institucional y la gobernanza del agua?*

---

**Análisis de los Archivos**

**1. SERVICIOS PÚBLICOS (⭐⭐⭐⭐⭐)**
Es el archivo más importante. Contiene indicadores clave que describen directamente la gobernanza del recurso hídrico (alineados con estándares del Banco Mundial y la OCDE):
* Cobertura de acueducto urbano / rural
* Cobertura de alcantarillado
* Continuidad del acueducto
* Calidad del agua y Acceso a agua potable
* Tratamiento de aguas residuales

**2. ECONOMÍA (❌)**
Solo contiene tres indicadores: *PIB*, *Exportaciones* e *Índice de desempeño institucional*.
* **Decisión:** Descartar PIB y Exportaciones porque no aportan valor predictivo para el LSTM.

**3. DESCRIPCIÓN GENERAL (⚠️)**
Contiene población, densidad, porcentajes urbano/rural, categoría municipal e *Índice de desempeño institucional*.
* **Hallazgo Clave:** El *Índice de Desempeño Institucional* está duplicado (aparece en Economía y en Descripción). Solo debemos cargarlo una vez.

---

**Propuesta: Creación de una Capa de Gobernanza Limpia**

En lugar de procesar más de 11.000 registros, propongo construir una capa depurada conservando únicamente lo relevante:

* **Indicadores a CONSERVAR (✅):** Acceso a agua potable, Calidad del agua, Cobertura acueducto (urbano/rural), Cobertura alcantarillado (urbano/rural), Continuidad del acueducto, Tratamiento de aguas residuales e Índice de desempeño institucional.
* **Indicadores a ELIMINAR (❌):** PIB, Exportaciones, Categoría Ley 617, Tipologías, Densidad, etc.

### Estructura sugerida para el dataset final:

| Nodo | Año | Indicador | Valor |
| :--- | :---: | :--- | :---: |
| Bogotá | 2022 | Cobertura Acueducto | 99.7 |
| Bogotá | 2022 | Continuidad | 24.0 |
| Bogotá | 2022 | Calidad Agua | 97.0 |
| Bogotá | 2022 | Tratamiento AR | 84.0 |
| Bogotá | 2022 | Desempeño Institucional | 89.0 |

*(Aplicando la misma estructura para **Chía**, **Tocancipá** y **Villapinzón**).*

---

## **Recomendación**

Sugiero **no iniciar la extracción definitiva todavía**. Primero hagamos una **auditoría técnica rápida** respondiendo:
1. ¿Qué años tiene cada indicador?
2. ¿Qué municipios de nuestro estudio (Bogotá, Chía, Tocancipá y Villapinzón) están disponibles?
3. ¿Cuántos registros existen para cada indicador en esos cuatro municipios?

Esta auditoría será una de las decisiones metodológicas más importantes del **Framework V7**, asegurando que la dimensión de Gobernanza represente datos reales, limpios y con verdadero valor analítico.

Auditoría 1. ¿Qué indicadores existen realmente?

In [ ]:
import pandas as pd

archivos = {
    "SERVICIOS":"https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/SERVICIOS_PUBLICOS.xlsx",
    "ECONOMIA":"https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/ECONOMIA.xlsx",
    "DESCRIPCION":"https://raw.githubusercontent.com/jriatiga/dataset/refs/heads/main/proyectoGradoV4/07_Capa_Gobernanza/DESCRIPCION_GENERAL.xlsx"
}

for nombre,ruta in archivos.items():

    print("="*90)
    print(nombre)
    print("="*90)

    df=pd.read_excel(ruta,header=11)

    resumen=(
        df.groupby("INDICADOR")
        .agg(
            Año_Inicial=("YEAR","min"),
            Año_Final=("YEAR","max"),
            Registros=("YEAR","count"),
            Municipios=("MUNICIPIO","nunique")
        )
        .sort_values("Registros",ascending=False)
    )

    print(resumen)

SERVICIOS
                                                    Año_Inicial  Año_Final  \
INDICADOR                                                                    
ACCESO A AGUA POTABLE ADECUADO                             2021       2024   
COBERTURA DE ACUEDUCTO URBANO                              2021       2024   
ACUEDUCTO EN AREA CENTROS POBLADOS                         2021       2024   
ACUEDUCTO EN AREA MUNICIPAL O DISTRITAL                    2021       2024   
ACUEDUCTO EN AREA URBANA                                   2021       2024   
ACUEDUCTO EN AREA RURAL (CENTROS POBLADOS Y FIN...         2021       2024   
ALCANTARILLADO EN AREA RURAL (CENTROS POBLADOS ...         2021       2024   
ALCANTARILLADO EN AREA MUNICIPAL O DISTRITAL               2021       2024   
ALCANTARILLADO EN AREA URBANA                              2021       2024   
ALCANTARILLADO EN AREA CENTROS POBLADOS                    2021       2024   
COBERTURA DE ALCANTARILLADO RURAL                     

Auditoría 2. ¿Qué información existe para nuestros municipios?

In [ ]:
municipios=[
    "BOGOTA",
    "CHIA",
    "TOCANCIPA",
    "VILLAPINZON",
    "SOPO"
]

for nombre,ruta in archivos.items():

    print("\n")
    print("="*90)
    print(nombre)
    print("="*90)

    df=pd.read_excel(ruta,header=11)

    df["MUNICIPIO"]=df["MUNICIPIO"].str.upper()

    filtro=df[df["MUNICIPIO"].isin(municipios)]

    print(filtro.groupby(["INDICADOR","MUNICIPIO"]).size())



SERVICIOS
INDICADOR                                                                 MUNICIPIO  
ACCESO A AGUA POTABLE ADECUADO                                            CHIA           4
                                                                          SOPO           4
                                                                          TOCANCIPA      4
                                                                          VILLAPINZON    4
ACUEDUCTO EN AREA CENTROS POBLADOS                                        CHIA           4
                                                                                        ..
PORCENTAJE DE AGUAS RESIDUALES INDUSTRIALES TRATADAS                      VILLAPINZON    2
PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS  CHIA           4
                                                                          SOPO           4
                                                                          TOCANCIPA

Auditoría 3. ¿Cuántos valores faltan?

In [ ]:
for nombre,ruta in archivos.items():

    print("\n")
    print("="*90)
    print(nombre)
    print("="*90)

    df=pd.read_excel(ruta,header=11)

    df["MUNICIPIO"]=df["MUNICIPIO"].str.upper()

    filtro=df[df["MUNICIPIO"].isin(municipios)]

    tabla=pd.pivot_table(

        filtro,

        index="INDICADOR",

        columns="MUNICIPIO",

        values="DATO_NUMERICO",

        aggfunc="count"

    )

    print(tabla.fillna(0))



SERVICIOS
MUNICIPIO                                           CHIA  SOPO  TOCANCIPA  \
INDICADOR                                                                   
ACCESO A AGUA POTABLE ADECUADO                         4     4          4   
ACUEDUCTO EN AREA CENTROS POBLADOS                     4     4          4   
ACUEDUCTO EN AREA MUNICIPAL O DISTRITAL                4     4          4   
ACUEDUCTO EN AREA RURAL (CENTROS POBLADOS Y FIN...     4     4          4   
ACUEDUCTO EN AREA URBANA                               4     4          4   
AGUAS RESIDUALES TRATADAS                              2     2          2   
ALCANTARILLADO EN AREA CENTROS POBLADOS                4     4          4   
ALCANTARILLADO EN AREA MUNICIPAL O DISTRITAL           4     4          4   
ALCANTARILLADO EN AREA RURAL (CENTROS POBLADOS ...     4     4          4   
ALCANTARILLADO EN AREA URBANA                          4     4          4   
CALIDAD DEL AGUA                                       2     2  

# Selección Definitiva: Capa de Gobernanza

¡Excelente elección! Reducir el set a estos **9 indicadores específicos** le da al modelo una dimensión de gobernanza e infraestructura sumamente sólida, limpia y sin ruido analítico.

Así queda nuestra selección final para la capa de capacidad institucional y gestión hídrica:

| Indicador | ¿Conservar? | Dimensión Representada |
| :--- | :---: | :--- |
| **Índice de Desempeño Institucional** | ✅ | Capacidad Administrativa y Gobernanza |
| **Acceso a Agua Potable Adecuado** | ✅ | Calidad de Vida y Salud Pública |
| **Cobertura Acueducto Urbano** | ✅ | Infraestructura Básica de Distribución |
| **Cobertura Acueducto Rural** | ✅ | Equidad Territorial y Acceso |
| **Cobertura Alcantarillado Urbano** | ✅ | Saneamiento y Control Sanitario Urbano |
| **Cobertura Alcantarillado Rural** | ✅ | Saneamiento y Control Sanitario Rural |
| **Continuidad Acueducto Urbano** | ✅ | Confiabilidad e Continuidad del Servicio |
| **Aguas Residuales Tratadas** | ✅ | Mitigación de Impacto Ambiental |
| **Acceso a Métodos de Saneamiento** | ✅ | Cobertura Sanitaria Global |

---

## 📌 Próximo Paso: Auditoría de Datos

Con esta lista de 9 indicadores clave ya definida, el siguiente paso es ejecutar la consulta de control en nuestro entorno para verificar la viabilidad de la serie de tiempo en nuestros nodos de interés (**Bogotá, Chía, Tocancipá y Villapinzón**):

1. **Ventana Temporal:** Identificar el rango de años real (ej. 2018-2024) con datos consistentes para este grupo.
2. **Densidad de Datos:** Validar que no existan vacíos (*gaps*) severos en los registros de estos municipios que puedan afectar el entrenamiento del modelo.
3. **Imputación:** En caso de que falten años específicos para algún municipio, definir la estrategia metodológica (ej. interpolación lineal o arrastre del último valor conocido).

1. SERVICIOS PUBLICOS

De los 18 indicadores conservaría solamente aquellos relacionados con gestión del agua.

Conservar
ACCESO A AGUA POTABLE ADECUADO
CALIDAD DEL AGUA
CONTINUIDAD DE ACUEDUCTO URBANO
COBERTURA DE ACUEDUCTO URBANO
COBERTURA DE ACUEDUCTO RURAL
COBERTURA DE ALCANTARILLADO URBANO
COBERTURA DE ALCANTARILLADO RURAL
AGUAS RESIDUALES TRATADAS
PORCENTAJE DE AGUAS RESIDUALES INDUSTRIALES TRATADAS
PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS

Eliminaría:

ACUEDUCTO EN AREA MUNICIPAL O DISTRITAL
ACUEDUCTO EN AREA URBANA
ACUEDUCTO EN AREA CENTROS POBLADOS
ACUEDUCTO EN AREA RURAL...
ALCANTARILLADO EN AREA MUNICIPAL...
ALCANTARILLADO EN AREA URBANA
ALCANTARILLADO EN AREA CENTROS POBLADOS
ALCANTARILLADO EN AREA RURAL...

porque son desagregaciones administrativas de las coberturas y aportan información redundante.

In [ ]:
indicadores_servicios = [

    "ACCESO A AGUA POTABLE ADECUADO",

    "CALIDAD DEL AGUA",

    "CONTINUIDAD DE ACUEDUCTO URBANO",

    "COBERTURA DE ACUEDUCTO URBANO",

    "COBERTURA DE ACUEDUCTO RURAL",

    "COBERTURA DE ALCANTARILLADO URBANO",

    "COBERTURA DE ALCANTARILLADO RURAL",

    "AGUAS RESIDUALES TRATADAS",

    "PORCENTAJE DE AGUAS RESIDUALES INDUSTRIALES TRATADAS",

    "PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS"

]

servicios = dfs['servicios']
servicios = servicios[
    servicios["INDICADOR"].isin(indicadores_servicios)
].copy()

print("Indicadores conservados:", servicios["INDICADOR"].nunique())

Indicadores conservados: 10


2. ECONOMIA

Aquí solamente existe un indicador realmente útil.

Conservar:

PRODUCTO INTERNO BRUTO DEPARTAMENTAL

Eliminar

EXPORTACIONES DE BIENES NO MINERO ENERGETICOS
INDICE DE DESEMPENO INSTITUCIONAL

porque sólo existen cuatro registros departamentales y no permiten análisis municipal.

In [ ]:
economia = dfs['economia']
economia = economia[
    economia["INDICADOR"] ==
    "PRODUCTO INTERNO BRUTO DEPARTAMENTAL"
].copy()

OBSERVACIÓN METODOLÓGICA

Durante el proceso de construcción de la Capa de Gobernanza se realizó una auditoría integral de los conjuntos de datos abiertos del portal de la Gobernación de Cundinamarca.

La depuración tuvo como objetivo garantizar la consistencia científica del Framework, conservando únicamente aquellos indicadores que:

• Presentan cobertura para los municipios de estudio (Chía, Tocancipá, Sopó y Villapinzón).
• Poseen series temporales comparables.
• Contienen variables cuantitativas susceptibles de análisis.
• Mantienen relación directa con la gobernanza del recurso hídrico, la prestación de servicios públicos, la capacidad institucional o el contexto socioeconómico.

Se eliminaron indicadores redundantes, clasificaciones administrativas y variables con cobertura insuficiente o exclusivamente departamental, reduciendo la dimensionalidad de la capa sin pérdida de información relevante para el análisis integrado del Framework.

Esta depuración mejora la interoperabilidad entre capas, disminuye la redundancia de información y fortalece la calidad analítica de los modelos posteriores de minería de datos y aprendizaje automático.

3. DESCRIPCION

Aquí conviene quedarse únicamente con variables estructurales.

Conservar

POBLACION TOTAL
DENSIDAD POBLACIONAL
PORCENTAJE POBLACION URBANA
PORCENTAJE POBLACION RURAL
INDICE DE DESEMPENO INSTITUCIONAL

Eliminar

TOTAL POBLACION

porque es duplicado de POBLACION TOTAL.

Eliminar

TIPOLOGIAS...

porque sólo existe un año.

Eliminar

CATEGORIA LEY 617

porque es una clasificación administrativa.

In [ ]:
indicadores_descripcion = [

    "POBLACION TOTAL",

    "DENSIDAD POBLACIONAL",

    "PORCENTAJE POBLACION URBANA",

    "PORCENTAJE POBLACION RURAL",

    "INDICE DE DESEMPENO INSTITUCIONAL"

]

descripcion = dfs['descripcion']
descripcion = descripcion[
    descripcion["INDICADOR"].isin(indicadores_descripcion)
].copy()

print(descripcion["INDICADOR"].unique())

['POBLACION TOTAL' 'DENSIDAD POBLACIONAL' 'PORCENTAJE POBLACION URBANA'
 'PORCENTAJE POBLACION RURAL' 'INDICE DE DESEMPENO INSTITUCIONAL']


4. Verificación final

In [ ]:
print("="*70)
print("SERVICIOS")
print(servicios["INDICADOR"].value_counts())

print("="*70)
print("ECONOMIA")
print(economia["INDICADOR"].value_counts())

print("="*70)
print("DESCRIPCION")
print(descripcion["INDICADOR"].value_counts())

SERVICIOS
INDICADOR
COBERTURA DE ACUEDUCTO URBANO                                               468
ACCESO A AGUA POTABLE ADECUADO                                              468
COBERTURA DE ACUEDUCTO RURAL                                                464
COBERTURA DE ALCANTARILLADO RURAL                                           464
COBERTURA DE ALCANTARILLADO URBANO                                          463
PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS    457
PORCENTAJE DE AGUAS RESIDUALES INDUSTRIALES TRATADAS                        234
CALIDAD DEL AGUA                                                            232
AGUAS RESIDUALES TRATADAS                                                   232
CONTINUIDAD DE ACUEDUCTO URBANO                                             232
Name: count, dtype: int64
ECONOMIA
INDICADOR
PRODUCTO INTERNO BRUTO DEPARTAMENTAL    569
Name: count, dtype: int64
DESCRIPCION
INDICADOR
PORCENTAJE POBLACION URBANA          585
PO

Una recomendación adicional

Veo algo importante en tu Framework: hasta ahora todas las capas (climática, hidráulica y percepción) terminan con un registro por nodo y por periodo. Para mantener esa consistencia, en Gobernanza no dejaría decenas de indicadores como filas. En su lugar, construiría una matriz maestra donde cada fila sea un municipio-año y cada indicador seleccionado sea una columna (formato ancho). Esto facilitará mucho la integración con el dataset maestro y el entrenamiento de los modelos LSTM y demás algoritmos posteriores. Considero que es el siguiente paso lógico una vez finalizada esta depuración.

In [ ]:
print("### Servicios DataFrame Head:")
display(servicios.head())

### Servicios DataFrame Head:


,COD_DEP,DEPARTAMENTO,COD_MUN,MUNICIPIO,DIMENSION,SUBCATEGORIA,INDICADOR,DATO_NUMERICO,CUALITATIVO,YEAR,FUENTE,UNIDAD_DE_MEDIDA
0,25,CUNDINAMARCA,25019,ALBAN,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,97.56,NaN,2024,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
1,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,96.00,NaN,2021,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
2,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,96.00,NaN,2022,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
3,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,94.38,NaN,2023,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
4,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,94.38,NaN,2024,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE


In [ ]:
print("### Economia DataFrame Head:")
display(economia.head())

### Economia DataFrame Head:


,COD_DEP,DEPARTAMENTO,COD_MUN,MUNICIPIO,DIMENSION,SUBCATEGORIA,INDICADOR,DATO_NUMERICO,CUALITATIVO,YEAR,FUENTE,UNIDAD_DE_MEDIDA
0,25,CUNDINAMARCA,25148,CAPARRAPI,ECONOMIA,ECONOMIA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,180.669847,NaN,2020,DANE,MILES DE MILLONES DE PESOS
1,25,CUNDINAMARCA,25151,CAQUEZA,ECONOMIA,ECONOMIA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,350.312909,NaN,2020,DANE,MILES DE MILLONES DE PESOS
2,25,CUNDINAMARCA,25154,CARMEN DE CARUPA,ECONOMIA,ECONOMIA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,130.770163,NaN,2020,DANE,MILES DE MILLONES DE PESOS
3,25,CUNDINAMARCA,25168,CHAGUANI,ECONOMIA,ECONOMIA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,61.140139,NaN,2020,DANE,MILES DE MILLONES DE PESOS
4,25,CUNDINAMARCA,25175,CHIA,ECONOMIA,ECONOMIA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,3503.635640,NaN,2020,DANE,MILES DE MILLONES DE PESOS


In [ ]:
print("### Descripcion DataFrame Head:")
display(descripcion.head())

### Descripcion DataFrame Head:


,COD_DEP,DEPARTAMENTO,COD_MUN,MUNICIPIO,DIMENSION,SUBCATEGORIA,INDICADOR,DATO_NUMERICO,CUALITATIVO,YEAR,FUENTE,UNIDAD_DE_MEDIDA
340,25351,CUNDINAMARCA,25899,ZIPAQUIRA,DESCRIPCION GENERAL,DESCRIPCION GENERAL,POBLACION TOTAL,145842.0,NaN,2021,DANE,PERSONAS
341,25352,CUNDINAMARCA,25898,ZIPACON,DESCRIPCION GENERAL,DESCRIPCION GENERAL,POBLACION TOTAL,5121.0,NaN,2021,DANE,PERSONAS
342,25353,CUNDINAMARCA,25885,YACOPI,DESCRIPCION GENERAL,DESCRIPCION GENERAL,POBLACION TOTAL,13124.0,NaN,2021,DANE,PERSONAS
343,25354,CUNDINAMARCA,25878,VIOTA,DESCRIPCION GENERAL,DESCRIPCION GENERAL,POBLACION TOTAL,14289.0,NaN,2021,DANE,PERSONAS
344,25355,CUNDINAMARCA,25875,VILLETA,DESCRIPCION GENERAL,DESCRIPCION GENERAL,POBLACION TOTAL,29642.0,NaN,2021,DANE,PERSONAS


In [ ]:
indicadores_servicios = [

    "COBERTURA DE ACUEDUCTO URBANO",

    "COBERTURA DE ACUEDUCTO RURAL",

    "COBERTURA DE ALCANTARILLADO URBANO",

    "COBERTURA DE ALCANTARILLADO RURAL",

    "ACCESO A AGUA POTABLE ADECUADO",

    "PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS",

    "CALIDAD DEL AGUA",

    "CONTINUIDAD DE ACUEDUCTO URBANO",

    "AGUAS RESIDUALES TRATADAS"

]

In [ ]:
indicadores_economia = [

    "PRODUCTO INTERNO BRUTO DEPARTAMENTAL"

]

In [ ]:
indicadores_descripcion = [

    "POBLACION TOTAL",

    "DENSIDAD POBLACIONAL",

    "PORCENTAJE POBLACION URBANA",

    "PORCENTAJE POBLACION RURAL",

    "INDICE DE DESEMPENO INSTITUCIONAL"

]

In [ ]:
servicios = servicios[
    servicios["INDICADOR"].isin(indicadores_servicios)
].copy()

economia = economia[
    economia["INDICADOR"].isin(indicadores_economia)
].copy()

descripcion = descripcion[
    descripcion["INDICADOR"].isin(indicadores_descripcion)
].copy()

In [ ]:
print("Servicios:", servicios.shape)

print("Economía:", economia.shape)

print("Descripción:", descripcion.shape)

Servicios: (3480, 12)
Economía: (569, 12)
Descripción: (2228, 12)


In [ ]:
gobernanza = pd.concat(

    [

        servicios,

        economia,

        descripcion

    ],

    ignore_index=True

)

In [ ]:
print(gobernanza.shape)

display(gobernanza.head())

(6277, 12)


,COD_DEP,DEPARTAMENTO,COD_MUN,MUNICIPIO,DIMENSION,SUBCATEGORIA,INDICADOR,DATO_NUMERICO,CUALITATIVO,YEAR,FUENTE,UNIDAD_DE_MEDIDA
0,25,CUNDINAMARCA,25019,ALBAN,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,97.56,NaN,2024,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
1,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,96.00,NaN,2021,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
2,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,96.00,NaN,2022,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
3,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,94.38,NaN,2023,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE
4,25,CUNDINAMARCA,25035,ANAPOIMA,SERVICIOS PUBLICOS,COBERTURA POR AREA,PORCENTAJE DE LA POBLACION CON ACCESO A METODO...,94.38,NaN,2024,EMPRESAS PUBLICAS DE CUNDINAMARCA,PORCENTAJE


In [ ]:
print(gobernanza["MUNICIPIO"].unique()[:20])

print()

print(gobernanza.shape)

['CHIA' 'SOPO' 'TOCANCIPA' 'VILLAPINZON']

(216, 12)


In [ ]:
print(type(servicios))
print(type(economia))
print(type(descripcion))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [ ]:
try:
    print(gobernanza.shape)
except:
    print("No existe gobernanza")

(216, 12)


In [ ]:
municipios_framework = [

    "CHIA",

    "SOPO",

    "TOCANCIPA",

    "VILLAPINZON"

]

gobernanza_filtrada = gobernanza[
    gobernanza["MUNICIPIO"].isin(municipios_framework)
].copy()

print(gobernanza_filtrada.shape)

(216, 12)


In [ ]:
gobernanza_matriz = gobernanza_filtrada.pivot_table(
    index=["COD_MUN","MUNICIPIO","YEAR"],
    columns="INDICADOR",
    values="DATO_NUMERICO",
    aggfunc="first"
).reset_index()

In [ ]:
print("="*70)
print("DIMENSIONES")
print("="*70)

print(gobernanza_matriz.shape)

print()

print("="*70)
print("COLUMNAS")
print("="*70)

print(gobernanza_matriz.columns.tolist())

display(gobernanza_matriz.head())

DIMENSIONES
(20, 18)

COLUMNAS
['COD_MUN', 'MUNICIPIO', 'YEAR', 'ACCESO A AGUA POTABLE ADECUADO', 'AGUAS RESIDUALES TRATADAS', 'CALIDAD DEL AGUA', 'COBERTURA DE ACUEDUCTO RURAL', 'COBERTURA DE ACUEDUCTO URBANO', 'COBERTURA DE ALCANTARILLADO RURAL', 'COBERTURA DE ALCANTARILLADO URBANO', 'CONTINUIDAD DE ACUEDUCTO URBANO', 'DENSIDAD POBLACIONAL', 'INDICE DE DESEMPENO INSTITUCIONAL', 'POBLACION TOTAL', 'PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS', 'PORCENTAJE POBLACION RURAL', 'PORCENTAJE POBLACION URBANA', 'PRODUCTO INTERNO BRUTO DEPARTAMENTAL']


INDICADOR,COD_MUN,MUNICIPIO,YEAR,ACCESO A AGUA POTABLE ADECUADO,AGUAS RESIDUALES TRATADAS,CALIDAD DEL AGUA,COBERTURA DE ACUEDUCTO RURAL,COBERTURA DE ACUEDUCTO URBANO,COBERTURA DE ALCANTARILLADO RURAL,COBERTURA DE ALCANTARILLADO URBANO,CONTINUIDAD DE ACUEDUCTO URBANO,DENSIDAD POBLACIONAL,INDICE DE DESEMPENO INSTITUCIONAL,POBLACION TOTAL,PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS,PORCENTAJE POBLACION RURAL,PORCENTAJE POBLACION URBANA,PRODUCTO INTERNO BRUTO DEPARTAMENTAL
0,25175,CHIA,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.89,83.11,3503.635640
1,25175,CHIA,2021,100.0,NaN,NaN,100.0,100.0,61.07,84.66,NaN,1952.83,NaN,148415.0,99.00,16.28,83.72,4194.505649
2,25175,CHIA,2022,100.0,NaN,NaN,100.0,1.0,59.83,98.09,NaN,2014.87,NaN,153130.0,99.00,15.83,84.17,160.198863
3,25175,CHIA,2023,100.0,69.0,25.00,100.0,1.0,1.00,99.00,24.0,2082.34,NaN,158258.0,98.85,15.42,84.58,5711.022704
4,25175,CHIA,2024,100.0,69.0,36.32,100.0,1.0,1.00,99.00,24.0,2148.76,69.6,163306.0,98.85,15.03,84.97,6182.859140


In [ ]:
gobernanza_matriz = gobernanza_matriz[[
    "COD_MUN",
    "MUNICIPIO",
    "YEAR",
    "POBLACION TOTAL",
    "DENSIDAD POBLACIONAL",
    "PORCENTAJE POBLACION URBANA",
    "PORCENTAJE POBLACION RURAL",
    "PRODUCTO INTERNO BRUTO DEPARTAMENTAL",
    "INDICE DE DESEMPENO INSTITUCIONAL",
    "ACCESO A AGUA POTABLE ADECUADO",
    "PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS",
    "COBERTURA DE ACUEDUCTO URBANO",
    "COBERTURA DE ACUEDUCTO RURAL",
    "COBERTURA DE ALCANTARILLADO URBANO",
    "COBERTURA DE ALCANTARILLADO RURAL",
    "CONTINUIDAD DE ACUEDUCTO URBANO",
    "CALIDAD DEL AGUA",
    "AGUAS RESIDUALES TRATADAS"
]]

In [ ]:
gobernanza_matriz = gobernanza_matriz.sort_values(
    ["MUNICIPIO","YEAR"]
).reset_index(drop=True)

In [ ]:
gobernanza_matriz.rename(
    columns={
        "YEAR":"Anio"
    },
    inplace=True
)

In [ ]:
gobernanza_matriz.insert(
    0,
    "Nodo",
    gobernanza_matriz["MUNICIPIO"]
)

In [ ]:
print(gobernanza_matriz.shape)

display(gobernanza_matriz.head(15))

(20, 19)


INDICADOR,Nodo,COD_MUN,MUNICIPIO,Anio,POBLACION TOTAL,DENSIDAD POBLACIONAL,PORCENTAJE POBLACION URBANA,PORCENTAJE POBLACION RURAL,PRODUCTO INTERNO BRUTO DEPARTAMENTAL,INDICE DE DESEMPENO INSTITUCIONAL,ACCESO A AGUA POTABLE ADECUADO,PORCENTAJE DE LA POBLACION CON ACCESO A METODOS DE SANEAMIENTO ADECUADOS,COBERTURA DE ACUEDUCTO URBANO,COBERTURA DE ACUEDUCTO RURAL,COBERTURA DE ALCANTARILLADO URBANO,COBERTURA DE ALCANTARILLADO RURAL,CONTINUIDAD DE ACUEDUCTO URBANO,CALIDAD DEL AGUA,AGUAS RESIDUALES TRATADAS
0,CHIA,25175,CHIA,2020,NaN,NaN,83.11,16.89,3503.635640,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CHIA,25175,CHIA,2021,148415.0,1952.83,83.72,16.28,4194.505649,NaN,100.00,99.00,100.0,100.0,84.66,61.07,NaN,NaN,NaN
2,CHIA,25175,CHIA,2022,153130.0,2014.87,84.17,15.83,160.198863,NaN,100.00,99.00,1.0,100.0,98.09,59.83,NaN,NaN,NaN
3,CHIA,25175,CHIA,2023,158258.0,2082.34,84.58,15.42,5711.022704,NaN,100.00,98.85,1.0,100.0,99.00,1.00,24.0,25.000,69.0
4,CHIA,25175,CHIA,2024,163306.0,2148.76,84.97,15.03,6182.859140,69.6,100.00,98.85,1.0,100.0,99.00,1.00,24.0,36.320,69.0
5,SOPO,25758,SOPO,2020,NaN,NaN,72.21,27.79,1137.646002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SOPO,25758,SOPO,2021,28868.0,255.47,73.18,26.82,1361.974553,NaN,100.00,99.00,100.0,99.0,77.88,29.02,NaN,NaN,NaN
7,SOPO,25758,SOPO,2022,29787.0,263.60,73.95,26.06,493.368277,NaN,100.00,99.00,1.0,99.0,100.00,29.02,NaN,NaN,NaN
8,SOPO,25758,SOPO,2023,30780.0,272.39,74.62,25.38,1854.394354,NaN,99.80,25.38,1.0,100.0,98.00,1.00,24.0,38.000,100.0
9,SOPO,25758,SOPO,2024,31775.0,281.19,75.23,24.77,2007.601733,88.7,99.84,98.77,1.0,100.0,98.00,1.00,24.0,68.920,100.0


In [ ]:
gobernanza_matriz.to_excel(
    "01_Capa_Gobernanza_V1.xlsx",
    index=False
)

print("Archivo generado correctamente.")

Archivo generado correctamente.


Una observación importante

A diferencia de la capa Climática (mensual) o la Hidráulica (mensual), esta capa de Gobernanza es anual. Esto no es un problema; simplemente significa que en la integración del Framework cada valor anual se asociará al periodo correspondiente mediante el campo Anio. Más adelante, cuando construyamos el dataset maestro para los modelos LSTM, definiremos la estrategia para armonizar las distintas resoluciones temporales (mensual vs. anual) sin perder consistencia.